# Linearized transport coordinates from a disk reference

This notebook generates `fig:monge-linearized-transport-coordinates`.  A fixed reference law $\rho$ is sampled uniformly on a disk.  For each target law $\mu$, POT computes a quadratic OT plan and its barycentric projection $T_\mu$.  The plotted vector field is the linearized OT coordinate
$$
T_\mu-\mathrm{Id}\in L^2(\rho;\mathbb R^2).
$$
The OT map is computed on a relatively dense cloud; only a farthest-point subset of arrows is displayed.

In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd(), Path.cwd() / "notebooks-figures", Path.cwd().parent / "notebooks-figures"]:
    if (candidate / "figure_style.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        ROOT = candidate.parent if candidate.name == "notebooks-figures" else candidate
        break
else:
    raise RuntimeError("Could not locate figure_style.py")

import matplotlib.pyplot as plt
import numpy as np
import ot
from scipy.ndimage import gaussian_filter

from figure_style import RED, VIOLET, BLUE, LIGHT_GRAY, figure_dir, interp_color, remove_axes, save_pdf, setup_matplotlib

setup_matplotlib()
rng = np.random.default_rng(20240608)


## Reference disk, target mixtures, and barycentric maps

The two targets are Gaussian mixtures with different locations and shapes.  We display the target density by level sets and overlay arrows from selected reference particles.

In [ ]:
def uniform_disk(n):
    theta = rng.uniform(0, 2*np.pi, n)
    radius = np.sqrt(rng.uniform(0, 1, n))
    return np.column_stack([radius * np.cos(theta), radius * np.sin(theta)]) * 0.58


def mixture_sample(n, weights, means, covs):
    weights = np.asarray(weights, dtype=float)
    weights = weights / weights.sum()
    counts = rng.multinomial(n, weights)
    pts = []
    for count, mean, cov in zip(counts, means, covs):
        pts.append(rng.multivariate_normal(mean, cov, size=count))
    out = np.vstack(pts)
    rng.shuffle(out)
    return out


def gaussian_density_grid(means, covs, weights, lim=1.55, m=140):
    xs = np.linspace(-lim, lim, m)
    ys = np.linspace(-lim, lim, m)
    X, Y = np.meshgrid(xs, ys)
    pts = np.stack([X, Y], axis=-1)
    Z = np.zeros_like(X)
    for w, mean, cov in zip(weights, means, covs):
        inv = np.linalg.inv(cov)
        det = np.linalg.det(cov)
        diff = pts - mean
        expo = np.einsum('...i,ij,...j->...', diff, inv, diff)
        Z += w * np.exp(-0.5 * expo) / (2*np.pi*np.sqrt(det))
    return xs, ys, Z


def barycentric_map(x, y):
    a = np.ones(len(x)) / len(x)
    b = np.ones(len(y)) / len(y)
    C = ot.dist(x, y, metric="sqeuclidean")
    P = ot.emd(a, b, C, numItermax=300000)
    return (P @ y) / a[:, None]


def farthest_subset(points, n):
    chosen = [int(np.argmin(np.sum(points**2, axis=1)))]
    dist2 = np.sum((points - points[chosen[0]])**2, axis=1)
    for _ in range(1, n):
        i = int(np.argmax(dist2))
        chosen.append(i)
        dist2 = np.minimum(dist2, np.sum((points - points[i])**2, axis=1))
    return np.asarray(chosen)

n = 300
rho = uniform_disk(n)

red_spec = dict(
    weights=[0.48, 0.32, 0.20],
    means=[[-0.92, 0.58], [-0.78, -0.50], [-0.18, 0.05]],
    covs=[[[0.035, 0.010], [0.010, 0.065]], [[0.055, -0.015], [-0.015, 0.035]], [[0.030, 0.0], [0.0, 0.030]]],
)
purple_spec = dict(
    weights=[0.42, 0.36, 0.22],
    means=[[0.82, 0.58], [0.95, -0.42], [0.22, -0.05]],
    covs=[[[0.050, -0.012], [-0.012, 0.032]], [[0.035, 0.014], [0.014, 0.060]], [[0.028, 0.0], [0.0, 0.040]]],
)

targets = {
    "red-mixture": (RED, red_spec, mixture_sample(n, **red_spec)),
    "purple-mixture": (VIOLET, purple_spec, mixture_sample(n, **purple_spec)),
}

arrow_idx = farthest_subset(rho, 48)


## Exported panels

Each panel shows the same disk reference.  The target density is shown by colored level sets; arrows display the barycentric projection of the OT map on a readable subsample.

In [ ]:
NAME = "monge-linearized-transport-coordinates"
OUT = figure_dir(NAME)

for filename, (color, spec, target) in targets.items():
    T = barycentric_map(rho, target)
    xs, ys, Z = gaussian_density_grid(spec['means'], spec['covs'], spec['weights'])

    fig, ax = plt.subplots(figsize=(2.15, 2.15))
    levels = np.linspace(Z.max() * 0.08, Z.max() * 0.88, 8)
    ax.contour(xs, ys, Z, levels=levels, colors=[color], linewidths=0.52, alpha=0.82)
    ax.scatter(rho[:, 0], rho[:, 1], s=6.0, color=LIGHT_GRAY, edgecolor="none", linewidth=0, zorder=2)
    ax.scatter(target[:, 0], target[:, 1], s=5.0, color=color, edgecolor="none", linewidth=0, alpha=0.55, zorder=2)
    for i in arrow_idx:
        start = rho[i]
        end = T[i]
        ax.annotate("", xy=end, xytext=start, arrowprops=dict(arrowstyle="->", color=color, lw=0.55, alpha=0.74, shrinkA=0, shrinkB=0), zorder=3)
    ax.set_xlim(-1.45, 1.45)
    ax.set_ylim(-1.28, 1.28)
    ax.set_aspect("equal")
    remove_axes(ax)
    save_pdf(fig, OUT / f"{filename}.pdf", pad_inches=0.035)
    plt.close(fig)
